# rel-hm Exploratory Data Analysis

Build an analysis-ready table from the RelBench H&M relational dataset, then run the shared EDA utility suite for profiling, distributions, relationships, outliers, and missing data.

## 1. Environment Setup

### 1.1 Load libraries and reusable EDA helpers

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join("..", "useful-python-scripts-eda")))
from data_profiler import DataProfiler
from distribution_analyzer import DistributionAnalyzer
from correlation_explorer import CorrelationExplorer
from outlier_suite import OutlierSuite
from missing_data_analyzer import MissingDataAnalyzer

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

from relbench.datasets import get_dataset

## 2. Dataset Loading

### 2.1 Download and instantiate the rel-hm database

In [ ]:
db = get_dataset("rel-hm", download=True).get_db(upto_test_timestamp=False)

### 2.2 Extract source tables

In [ ]:
articles_df = db.table_dict["article"].df
customers_df = db.table_dict["customer"].df
transactions_df = db.table_dict["transactions"].df

## 3. Analysis Table Construction

### 3.1 Join transactions with article and customer attributes

In [ ]:
articles_cols = ["article_id", "product_group_name", "colour_group_name", "section_name"]
customers_cols = [col for col in customers_df.columns if col not in ["postal_code", "fashion_news_frequency"]]

df = transactions_df.merge(customers_df[customers_cols], on="customer_id", how="left")
df = df.merge(articles_df[articles_cols], on="article_id", how="left")

In [ ]:
# If stated by the user, a sample of the dataset is used.
SAMPLE_SIZE = 50000
if SAMPLE_SIZE is not None and len(df) > SAMPLE_SIZE:
    df = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)

### 3.2 Cast categorical identifier fields

In [ ]:
# Some columns are clearly categorical, so we transform them to string datatype.
categorical_cols = ["customer_id", "article_id", "sales_channel_id", "FN", "Active"]
for col in categorical_cols:
    df[col] = df[col].astype(str)

### 3.3 Preview the master analysis table

In [ ]:
df.head(5)

## 4. Comprehensive Data Profiling

### 4.1 Initialize the profiler

In [ ]:
# Create profiler.
profiler = DataProfiler(df, high_cardinality_threshold=0.5)

### 4.2 Print the profile summary

In [ ]:
# Print summary to console.
profiler.print_summary()

### 4.3 Generate the full profile report

In [ ]:
# Get detailed report.
profiler_report = profiler.generate_full_profile()
print(profiler_report["overview"])
print("--" * 20)
print(profiler_report["numeric_profiles"])
print("--" * 20)
print(profiler_report["categorical_profiles"])
print("--" * 20)
print(profiler_report["data_quality_issues"])

## 5. Distribution Analysis

### 5.1 Initialize the distribution analyzer

In [ ]:
analyzer = DistributionAnalyzer(df, exclude_columns=["customer_id", "article_id", "sales_channel_id"])

### 5.2 Generate distribution statistics

In [ ]:
# Generate distribution report
analyzer_report = analyzer.generate_distribution_report()
print(analyzer_report)

### 5.3 Identify highly skewed variables

In [ ]:
# Identify highly skewed columns
skewed = analyzer_report[abs(analyzer_report['skewness']) > 2]
print(f"Highly skewed columns: {skewed['column'].tolist()}")

### 5.4 Visualize numeric and categorical distributions

In [ ]:
# Visualize distributions
analyzer.plot_numeric_distributions(max_cols=10)
analyzer.plot_boxplots()
analyzer.plot_qq_plots()
analyzer.plot_categorical_distributions()

## 6. Correlation and Relationship Exploration

### 6.1 Initialize the relationship explorer

In [ ]:
explorer = CorrelationExplorer(df, exclude_columns=["customer_id", "article_id", "sales_channel_id"])

### 6.2 Find strongly correlated feature pairs

In [ ]:
# Find high correlations
high_corr = explorer.find_high_correlations(threshold=0.7, method='pearson')
print(high_corr)

### 6.3 Check multicollinearity with VIF

In [ ]:
# Check for multicollinearity
vif = explorer.calculate_vif()
problematic = vif[vif['vif'] > 10]
print(f"Features with high multicollinearity:\n{problematic}")

### 6.4 Run mutual information analysis when a target exists

In [ ]:
# Mutual information with target
if 'target' in df.columns:
    mi_scores = explorer.mutual_information_analysis('target')
    print(f"Top features:\n{mi_scores.head(10)}")

### 6.5 Visualize pairwise relationships

In [ ]:
# Visualize
explorer.plot_correlation_heatmap(method='pearson')
explorer.plot_correlation_comparison()
explorer.plot_scatter_matrix(max_cols=5)
explorer.plot_top_correlations(n_pairs=10)

## 7. Outlier Detection

### 7.1 Initialize the outlier analysis suite

In [ ]:
suite = OutlierSuite(df, exclude_columns=["customer_id", "article_id", "sales_channel_id"])

### 7.2 Compare outlier detection methods

In [ ]:
# Compare methods across all columns
summary = suite.compare_methods_all_columns()
print(summary)

### 7.3 Inspect price outliers

In [ ]:
# Analyze specific column
col = "price"
suite.plot_outlier_comparison(col)

### 7.4 Detect and visualize multivariate outliers

In [ ]:
# Detect multivariate outliers
iso_outliers = suite.detect_isolation_forest_outliers(contamination=0.1)
print(f"Found {iso_outliers.sum()} multivariate outliers")

feature1 = "price"
feature2 = "age"
suite.plot_multivariate_outliers([feature1, feature2])

### 7.5 Estimate the impact of price outliers

In [ ]:
# Analyze outlier impact
col = "price"
impact = suite.analyze_outlier_impact(col)
print(impact)

## 8. Missing Data Analysis

### 8.1 Initialize the missing data analyzer

In [ ]:
analyzer = MissingDataAnalyzer(df, exclude_columns=["customer_id", "article_id", "sales_channel_id"])

### 8.2 Generate missingness summaries and recommendations

In [ ]:
# Generate full report
report = analyzer.generate_full_report()

print("Missing Value Summary:")
print(report['summary'])

print("\nMissingness Patterns (co-occurrence):")
print(report['patterns'])

print("\nMissingness Classifications:")
print(report['classifications'])

print("\nImputation Recommendations:")
print(report['recommendations'])

### 8.3 Visualize missing data patterns

In [ ]:
# Visualize patterns
if report['summary'].empty:
    print("No missing values to visualize.")
else:
    analyzer.plot_missing_bar()
    analyzer.plot_missing_heatmap(max_cols=30)
    analyzer.plot_missing_correlation()

### 8.4 Classify age missingness and choose an imputation strategy

In [ ]:
# Classify specific column
col = "age"
if report['summary'].empty:
    print("No columns with missing values to classify or impute.")
elif col not in report['summary']['column'].values:
    print(f"Column '{col}' has no missing values to classify or impute.")
else:
    classification = analyzer.classify_missingness_type(col)
    recommendation = analyzer.recommend_strategy(col)
    print(f"Missingness type: {classification['missingness_type']}")
    print(f"Recommendation: {recommendation}")